# EDA analysis

**Author** : Vukasin Jovanovic

**Date** : 23.11.2026

## Objective

The goal of this project is to build a regression model that predicts the price of a used car based on its characteristics (e.g., brand, model, year, mileage, engine, fuel type).

This notebook covers the **Exploratory Data Analysis (EDA)** phase. Its purpose is to:

- Understand the structure and content of the dataset
- Assess data quality (missing values, duplicates, invalid or inconsistent entries)
- Analyze the distribution of the target variable (`price`)
- Identify the features most strongly related to price
- Detect outliers, skewness and multicollinearity that may affect modeling

The findings from this analysis will guide the next steps: **data cleaning**, **feature engineering** and **model selection**.

## Setup

In [11]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None) # show all columns in head()
pd.set_option("display.float_format", "{:,.2f}".format) # readable numbers in describe()

sns.set_theme(style="whitegrid", palette="deep")  # consistent style for all plots
plt.rcParams["figure.figsize"] = (10, 6) # default plot size

In [12]:
df = pd.read_csv('../data/raw/cars.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 56244 entries, 0 to 56243
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   make                 56244 non-null  str    
 1   model                56244 non-null  str    
 2   priceUSD             56244 non-null  int64  
 3   year                 56244 non-null  int64  
 4   condition            56244 non-null  str    
 5   mileage(kilometers)  56244 non-null  float64
 6   fuel_type            56244 non-null  str    
 7   volume(cm3)          56197 non-null  float64
 8   color                56244 non-null  str    
 9   transmission         56244 non-null  str    
 10  drive_unit           54339 non-null  str    
 11  segment              50953 non-null  str    
dtypes: float64(2), int64(2), str(8)
memory usage: 5.1 MB


### Observations
- The dataset has 56,244 rows and 12 columns: 4 numerical and 8 categorical.
- The target variable `priceUSD` has no missing values.
- Three columns have missing values:
    - `volume(cm3)`: 47 (0.08%)
    - `drive_unit`: 1,905 (3.4%)
    - `segment`: 5,291 (9.4%)

    The first two are small enough for simple imputation or row removal. `segment` needs a closer look during data cleaning.

- Column names mix camelCase and units in parentheses, so I'll rename them to snake_case.
- `mileage(kilometers)` is stored as `float`, even though mileage should be a whole number. Next I'll check whether any values have a fractional part.

In [27]:
df.columns = [x.lower().replace('(', '_').replace(')',' ').strip().replace(' ','_') for x in df.columns]
df = df.rename(columns={"priceusd" : "price_usd"})

In [30]:
display(df.head())

df.describe().T

,make,model,price_usd,year,condition,mileage_kilometers,fuel_type,volume_cm3,color,transmission,drive_unit,segment
0,mazda,2,5500,2008,with mileage,"162,000.00",petrol,"1,500.00",burgundy,mechanics,front-wheel drive,B
1,mazda,2,5350,2009,with mileage,"120,000.00",petrol,"1,300.00",black,mechanics,front-wheel drive,B
2,mazda,2,7000,2009,with mileage,"61,000.00",petrol,"1,500.00",silver,auto,front-wheel drive,B
3,mazda,2,3300,2003,with mileage,"265,000.00",diesel,"1,400.00",white,mechanics,front-wheel drive,B
4,mazda,2,5200,2008,with mileage,"97,183.00",diesel,"1,400.00",gray,mechanics,front-wheel drive,B


,count,mean,std,min,25%,50%,75%,max
price_usd,"56,244.00","7,415.46","8,316.96",48.00,"2,350.00","5,350.00","9,807.50","235,235.00"
year,"56,244.00","2,003.45",8.14,"1,910.00","1,998.00","2,004.00","2,010.00","2,019.00"
mileage_kilometers,"56,244.00","244,395.63","321,030.67",0.00,"137,000.00","228,500.00","310,000.00","9,999,999.00"
volume_cm3,"56,197.00","2,104.86",959.20,500.00,"1,600.00","1,996.00","2,300.00","20,000.00"


### Observations

- `price_usd` ranges from 48 to 235,235 USD. A price of 48 USD is unrealistically low for a car, so I'll look at the cheapest listings. The maximum may be a typo (repeating digits) or a genuine luxury car. The mean (7,415) is well above the median (5,350), so the distribution is right-skewed, and a log transformation of the target may help later.
- `year` has a minimum of 1910. This could be a vintage car or an entry error, and I'll inspect the oldest listings.
- `mileage_kilometers` ranges from 0 to 9,999,999 km. The maximum is almost certainly a placeholder value, and it also inflates the mean (244,396 km). Zero mileage is plausible only for new cars, so I'll compare it with the `condition` column. The column is stored as `float`, although the values in the first rows are whole numbers. I'll check whether any fractional values exist.
- `volume_cm3` has a maximum of 20,000 cm³, which is not realistic for a passenger car and is likely an entry error. I'll look at the largest values before deciding how to handle them.